# Fault-Code Vision — CUDA, Training & MLOps Playbook

**Adjoins:** [`fault_code_gan_synthetic_images.ipynb`](./fault_code_gan_synthetic_images.ipynb) (lab: GAN theory + synthetic displays)
**Package:** `ml/fault_code_vision/`
**Domain:** WarrantyGraph — claim photo of machine fault code → OCR → Neo4j GraphRAG

---

## What this notebook is

The GAN notebook proves **image synthesis**. This playbook answers the client question:

> *If we need CUDA, trained models, and full MLOps — what do we build, what do we need, and how does it work?*

It is **executable**: environment checks, dataset bootstrap, OCR train/eval, Cypher bridge, registry/eval floors, and a delivery checklist — all in-repo.

---

## Mental model (do not confuse these three systems)

| System | Job | When | Production? |
|--------|-----|------|-------------|
| **A. Generator (GAN/cGAN/diffusion)** | Manufacture labelled display photos | Offline batch | Data factory only |
| **B. Reader (OCR / CNN / TrOCR)** | Pixels → fault code string | Online on claim | **Yes — production ML** |
| **C. GraphRAG diagnose** | Code → FM → steps/parts (Neo4j) | Online | **Yes — deterministic core (already built)** |

> We do **not** put a GAN on the live claim path. We train a **reader**, gate it, pin it in the registry, and feed `match_error_codes` / `INDICATES` boosts.

## Table of contents

| Part | Content |
|------|---------|
| **0** | Repo map & imports |
| **1** | How the full system works (train vs runtime) |
| **2** | CUDA / device readiness |
| **3** | Data strategy & manifest schema |
| **4** | Bootstrap seeds + train production **reader** (OCR CNN) |
| **5** | Eval gates & promotion floors |
| **6** | Cypher / GraphRAG bridge + API payload |
| **7** | Model registry, FinOps, observability |
| **8** | Client bill of materials & phased plan |
| **9** | Checklist + next commands |

### Authoritative hooks in this monorepo

| Concern | Path |
|---------|------|
| LLMOps module | `docs/sdd/09-PLATFORM-LLMOPS.md` |
| Model registry | `models/registry.yaml` (`fault-code-ocr`, `fault-code-gan` stubs) |
| Finetune record example | `models/finetunes/fault-code-ocr.example.yaml` |
| Vision eval | `evals/vision/`, floors in `evals/thresholds.yaml` |
| ML package | `ml/fault_code_vision/` |
| GPU Dockerfile | `docker/Dockerfile.ml` |
| ML deps | `requirements-ml.txt` |
| Diagnose Cypher | `graph/graph_rag.py` → `match_error_codes`, `rank_failure_modes_with_error_codes` |

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# Ensure repo root is on path whether kernel cwd is repo root or notebooks/
ROOT = Path(".").resolve()
if not (ROOT / "ml").is_dir():
    if (ROOT.parent / "ml").is_dir():
        ROOT = ROOT.parent
    elif (ROOT / "notebooks").is_dir() is False and (ROOT.parent.parent / "ml").is_dir():
        ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

import torch
from PIL import Image
import matplotlib.pyplot as plt

from ml.fault_code_vision.device import device_report, pick_device, assert_cuda_for_client_train
from ml.fault_code_vision.catalog import FAULT_CATALOG, N_CLASSES, catalog_codes
from ml.fault_code_vision.pipeline import bootstrap_demo_dataset, mlops_checklist
from ml.fault_code_vision.dataset import load_manifest
from ml.fault_code_vision.cypher_bridge import cypher_for_extracted_code, diagnose_payload_from_ocr
from ml.fault_code_vision.infer import FaultCodeReader, user_message_for_graph_rag

ART = ROOT / "notebooks" / "fault_code_gan_artifacts" / "mlops_playbook"
ART.mkdir(parents=True, exist_ok=True)
CKPT_DIR = ART / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

print("REPO ROOT:", ROOT)
print("Artifacts:", ART)
print(f"Catalog size: {N_CLASSES} codes ->", ", ".join(catalog_codes()))

## Part 1 — How it works end-to-end

### Training time (offline, prefer NVIDIA CUDA)

```text
Real claim photos (when any) ──┐
Procedural LCD seeds ──────────┼──► versioned dataset (manifest + object store)
Classical augments ────────────┤
GAN/diffusion synthetic ───────┘
         │
         ▼
   Train OCR / closed-set classifier (CUDA)
         │
         ▼
   Eval gates (synthetic hold-out + REAL hold-out)
         │
         ▼
   models/registry.yaml pin ──► deploy canary ──► full
```

### Runtime (online — usually CPU; GPU only if QPS needs it)

```text
Customer photo
  → authz + malware scan + size limits
  → FaultCodeReader (pinned artifact)
  → closed-set ∩ product HAS_ERROR_CODE
  → low conf? escalate / retake
  → else user_message or structured error_codes[]
  → graph_rag.match_error_codes + INDICATES boost
  → CONFIRMS steps + parts + provenance (incl. model_version)
```

### Design rule (ADR 0001 aligned)

- **Deterministic core** stays GraphRAG — vision is an **input normalizer**, not a free-form diagnostic LLM.
- Synthetic data is labelled `source=synthetic` and never silent “historical claim truth.”
- Promote OCR only when **real-photo** floors pass (see `evals/thresholds.yaml` → `vision_full.real_photo_accuracy`).

## Part 2 — CUDA / device readiness

Client “we need CUDA” means a **training environment**, not GPUs on every diagnose pod.

| Claim | Proof |
|-------|--------|
| CUDA ready | `nvidia-smi` green + `torch.cuda.is_available()==True` |
| Image ready | `docker/Dockerfile.ml` builds; `docker run --gpus all ...` |
| Train job | `python -m ml.fault_code_vision.train_ocr --require-cuda ...` |

Local Mac demos may use **MPS**; official client train target is **NVIDIA CUDA**.

In [ ]:
report = device_report()
print(json.dumps(report, indent=2))

device = pick_device()
print("pick_device() ->", device)

if report["cuda_available"]:
    print("OK: CUDA available for client-style training")
    # quick tensor smoke
    x = torch.randn(2, 3, device=device)
    print("tensor device:", x.device, "sum", float(x.sum()))
else:
    print("NOTE: CUDA not available in this environment.")
    print("  - Local demo: MPS/CPU is fine for the playbook.")
    print("  - Client train: use a GPU VM or docker/Dockerfile.ml with --gpus all")
    print("  - Install GPU PyTorch: https://pytorch.org (cu124 wheels etc.)")

# Client train entrypoints should use:
#   assert_cuda_for_client_train(allow_cpu_fallback=False)
try:
    d = assert_cuda_for_client_train(allow_cpu_fallback=True)
    print("assert_cuda_for_client_train(allow_cpu_fallback=True) ->", d)
except RuntimeError as exc:
    print("CUDA required failure:", exc)

## Part 3 — Data strategy

### Manifest schema (one row per image)

| Field | Meaning |
|-------|---------|
| `path` | Image path |
| `code` | Ground-truth fault code |
| `class_idx` | Catalog index |
| `split` | train / val / test |
| `source` | `real` \| `synthetic` \| `seed` \| `augment` |
| `product_id` | Product context for closed-set intersection |

### Considerations

1. **Zero real photos today** → seeds + GAN for development only.
2. **Closed-set per product** → reject codes not on `HAS_ERROR_CODE`.
3. **PII** → claim photos may show homes/faces; retention + access control.
4. **Lineage** → never mix synthetic into historical claim analytics unflagged.
5. **Real hold-out** → frozen set for promotion; never train on it.

In [ ]:
# Bootstrap a small stratified seed dataset for OCR train (dev proxy)
manifest_path = bootstrap_demo_dataset(ART / "dataset", n_per_code=24, image_size=64)
rows = load_manifest(manifest_path)
from collections import Counter
print("Manifest:", manifest_path)
print("N=", len(rows), "splits=", Counter(r["split"] for r in rows))
print("codes sample:", sorted({r["code"] for r in rows})[:8], "...")

# Preview a few seeds
fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))
for ax, row in zip(axes.ravel(), rows[:12]):
    ax.imshow(Image.open(row["path"]))
    ax.set_title(f"{row['code']}/{row['split']}", fontsize=8)
    ax.axis("off")
plt.suptitle("Bootstrap seed displays for OCR training")
plt.tight_layout()
plt.savefig(ART / "preview_bootstrap.png", dpi=120)
plt.show()

## Part 4 — Train the production reader (not the GAN)

The **GAN** (other notebook) augments sparse data.
The **reader** is what production calls.

Package entrypoint:

```bash
python -m ml.fault_code_vision.train_ocr \
  --manifest <manifest.json> \
  --epochs 20 \
  --out models/artifacts/fault-code-ocr/ocr-v1.pt \
  --require-cuda   # on client GPU
```

Below we train a short run in-process for the playbook (CPU/MPS/CUDA).

In [ ]:
from ml.fault_code_vision.train_ocr import main as train_ocr_main

ckpt_path = CKPT_DIR / "fault_code_ocr_playbook.pt"
# Interactive notebook default; raise to 20–40 + --require-cuda on client GPU
rc = train_ocr_main([
    "--manifest", str(manifest_path),
    "--epochs", "12",
    "--batch-size", "64",
    "--lr", "0.002",
    "--out", str(ckpt_path),
    # omit --require-cuda so local MPS/CPU works; add it on client GPU jobs
])
print("train exit code:", rc)
print("checkpoint:", ckpt_path, "exists=", ckpt_path.exists())
if ckpt_path.with_suffix(".metrics.json").exists():
    print(ckpt_path.with_suffix(".metrics.json").read_text()[:500], "...")

## Part 5 — Eval gates & promotion

| Suite | Floor (see `evals/thresholds.yaml`) | Use |
|-------|-------------------------------------|-----|
| `vision_smoke` | `code_accuracy` ≥ 0.80 | Dev / PR proxy on synthetic |
| `vision_full` | synthetic ≥ 0.90; **`real_photo_accuracy` ≥ 0.85** | Production pin |

**Never** lower floors to go green. **Never** promote on synthetic-only when real photos exist.

```bash
python -m ml.fault_code_vision.eval_vision \
  --checkpoint .../ocr.pt \
  --manifest .../ocr_train_manifest.json \
  --split test \
  --min-accuracy 0.80
```

In [ ]:
from ml.fault_code_vision.eval_vision import evaluate_checkpoint

report = evaluate_checkpoint(ckpt_path, manifest_path, split="test")
summary = {k: report[k] for k in ("n", "correct", "accuracy", "checkpoint")}
print(json.dumps(summary, indent=2))

floor = 0.80  # vision_smoke
status = "PASS" if report["accuracy"] >= floor else "FAIL"
print(f"{status}: accuracy {report['accuracy']:.3f} vs floor {floor:.3f}")

report_path = ART / "ocr_eval_report.json"
report_path.write_text(json.dumps(report, indent=2))
print("wrote", report_path)

# Spot-check predictions
reader = FaultCodeReader(ckpt_path)
fig, axes = plt.subplots(1, 6, figsize=(10, 2.2))
test_rows = [r for r in rows if r["split"] == "test"][:6]
for ax, row in zip(axes, test_rows):
    img = Image.open(row["path"])
    pred = reader.predict(img)
    ax.imshow(img)
    ok = "OK" if pred["code"] == row["code"] else "MISS"
    ax.set_title(f"{row['code']}→{pred['code']}\n{pred['confidence']:.2f} {ok}", fontsize=8)
    ax.axis("off")
plt.suptitle("OCR reader predictions on test split")
plt.tight_layout()
plt.savefig(ART / "preview_ocr_preds.png", dpi=120)
plt.show()

## Part 6 — Bridge into GraphRAG / Cypher

Production path (`graph/graph_rag.py`):

1. `match_error_codes(product_id, user_message)` — codes on product whose string appears in message.
2. `rank_failure_modes_with_error_codes` — `(:ErrorCode)-[:INDICATES]->(:FailureMode)` confidence boost.

Vision should emit either:

- free-text: `Error code 5E shown on machine.` (substring match), or
- structured: `extracted_error_codes: ["5E"]` (preferred API shape when you wire multipart upload).

In [ ]:
# End-to-end offline: image -> OCR -> API payload -> Cypher
demo = next(r for r in rows if r["code"] == "5E" and r["split"] == "test")
ocr = reader.predict(demo["path"], min_confidence=0.3)
payload = diagnose_payload_from_ocr(ocr, product_id="wm-001")

print("OCR:", {k: ocr[k] for k in ("code", "confidence", "accepted", "escalate")})
print("user_message:", payload.get("user_message"))
print("\n--- Cypher: failure_modes_for_code ---")
print(payload["cypher"]["failure_modes_for_code"])
print("params:", payload["cypher"]["params"])

# Show all templates
print("\nAvailable Cypher keys:", list(cypher_for_extracted_code("5E").keys()))

# Conceptual API contract (not mounted in FastAPI yet — integration work)
api_contract = {
    "endpoint": "POST /claims/{id}/attachments  OR  POST /diagnose multipart",
    "request": {"product_id": "wm-001", "image": "<bytes>", "asset_id": "optional"},
    "response_fields": [
        "extracted_code", "confidence", "model_version",
        "ranked_failure_modes", "diagnostic_steps", "should_escalate",
    ],
    "graph_hooks": [
        "graph.graph_rag.match_error_codes",
        "graph.graph_rag.rank_failure_modes_with_error_codes",
    ],
}
print("\nAPI contract sketch:")
print(json.dumps(api_contract, indent=2))
(ART / "api_contract_sketch.json").write_text(json.dumps(api_contract, indent=2))

## Part 7 — MLOps control plane (this repo)

### Lifecycle (memorize)

```text
Data → Train → Eval gates → Registry pin → Deploy (flag/alias)
  → Monitor quality/cost/latency → Rollback or retrain
```

### Artifacts already stubbed

| Artifact | Role |
|----------|------|
| `models/registry.yaml` → `fault-code-ocr` / `fault-code-gan` | Aliases (`status: inactive` until gated) |
| `models/finetunes/fault-code-ocr.example.yaml` | Provenance template |
| `evals/thresholds.yaml` → `vision_smoke` / `vision_full` | Floors |
| `evals/vision/` | Suite docs + schema examples |
| `docker/Dockerfile.ml` | CUDA train image |
| `requirements-ml.txt` | ML deps separated from core API |
| `finops/budget.py` | Extend with $/1k images + GPU-hours |
| `observability/` + Prometheus | Add `ocr_*` series at API wire-up |
| `guardrails/` | Rate limit, size caps, closed-set on upload |

### Rollback

Flip registry `artifact` / `status` to previous pin — same pattern as LLM alias rollback. No need to redeploy the graph.

In [ ]:
import yaml

reg_path = ROOT / "models" / "registry.yaml"
reg = yaml.safe_load(reg_path.read_text())
vision_aliases = {k: v for k, v in reg.get("models", {}).items() if k.startswith("fault-code")}
print("Vision registry stubs:")
print(yaml.dump(vision_aliases, sort_keys=False))

thr = yaml.safe_load((ROOT / "evals" / "thresholds.yaml").read_text())
print("Vision floors:")
print(yaml.dump({k: thr[k] for k in thr if k.startswith("vision")}, sort_keys=False))

# FinOps-style unit costs (illustrative — wire to finops/ when serving)
finops_sketch = {
    "train_gpu_hour_usd": 1.5,
    "infer_cpu_per_1k_images_usd": 0.05,
    "infer_gpu_per_1k_images_usd": 0.40,
    "daily_budget_usd_suggestion": 25.0,
    "metrics": [
        "ocr_requests_total",
        "ocr_latency_seconds",
        "ocr_confidence",
        "ocr_unknown_or_low_conf_total",
        "ocr_model_version",
    ],
}
print("FinOps / metrics sketch:")
print(json.dumps(finops_sketch, indent=2))
(ART / "finops_metrics_sketch.json").write_text(json.dumps(finops_sketch, indent=2))

## Part 8 — What you need (bill of materials)

### People

| Role | Responsibility |
|------|----------------|
| ML engineer | CUDA train, OCR/GAN experiments |
| MLOps / platform | Registry, GPU jobs, monitoring, canary |
| Backend | Upload API → reader → diagnose |
| Data steward | Labels, OEM code lists, privacy |
| Domain SME | Real photo review, false-positive risk |
| Security | Upload surface, retention, threat model |

### Infrastructure

| Piece | Purpose |
|-------|---------|
| NVIDIA GPU VM or on-prem | CUDA train (T4/L4/A10+) |
| Object storage | Images, datasets, `.pt` artifacts |
| Experiment tracker | MLflow / W&B / Azure ML (optional but recommended) |
| CI | Unit tests + vision eval on PR/nightly |
| Serving | In-process Torch or Triton/TorchServe sidecar |
| Existing | FastAPI, dual Neo4j, Prometheus, evals, registry |

### Client must provide

1. Any historical display photos (even 50–200)
2. Authoritative error-code lists per product family
3. Image retention / consent policy
4. Success criteria (e.g. ≥85% real-photo code accuracy)

### Phased delivery

| Phase | Outcome |
|-------|---------|
| **0** | Charter: reader vs generator vs GraphRAG; metrics |
| **1** | CUDA env + `Dockerfile.ml` green |
| **2** | Dataset platform + real photo collection starts |
| **3** | Train reader; synthetic + real eval |
| **4** | Registry pin + CI gate + canary |
| **5** | API integration + closed-set + escalate |
| **6** | Operate: drift, retrain, dashboards, model cards |

In [ ]:
import pandas as pd

checklist = mlops_checklist()
df = pd.DataFrame(checklist)
print(df.to_string(index=False))
df.to_csv(ART / "mlops_checklist.csv", index=False)
print("\nWrote", ART / "mlops_checklist.csv")

# Persist a one-page executive summary for client packs
summary = {
    "title": "Fault-code vision MLOps",
    "production_model": "fault-code-ocr (reader)",
    "offline_factory": "fault-code-gan / procedural seeds",
    "core_diagnosis": "Neo4j GraphRAG (unchanged architecture)",
    "cuda": "Required for official train jobs; optional for inference",
    "promotion_gate": "evals/thresholds.yaml vision_full.real_photo_accuracy",
    "package": "ml/fault_code_vision",
    "lab_notebook": "notebooks/fault_code_gan_synthetic_images.ipynb",
    "this_playbook": "notebooks/fault_code_vision_mlops_playbook.ipynb",
    "registry_aliases": ["fault-code-ocr", "fault-code-gan"],
}
(ART / "executive_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## Part 9 — Copy-paste commands (client GPU)

```bash
# From repo root
pip install -r requirements-ml.txt
# On NVIDIA hosts, install CUDA PyTorch per pytorch.org if needed

# 1) Data
python -c "from ml.fault_code_vision.pipeline import bootstrap_demo_dataset; \
print(bootstrap_demo_dataset('notebooks/fault_code_gan_artifacts/ocr_demo', n_per_code=40))"

# 2) Train (CUDA enforced)
python -m ml.fault_code_vision.train_ocr \
  --manifest notebooks/fault_code_gan_artifacts/ocr_demo/ocr_train_manifest.json \
  --epochs 30 \
  --out models/artifacts/fault-code-ocr/ocr-v1.pt \
  --require-cuda

# 3) Eval gate
python -m ml.fault_code_vision.eval_vision \
  --checkpoint models/artifacts/fault-code-ocr/ocr-v1.pt \
  --manifest notebooks/fault_code_gan_artifacts/ocr_demo/ocr_train_manifest.json \
  --split test \
  --min-accuracy 0.85 \
  --report evals/vision/reports/ocr-v1.json

# 4) Docker GPU train
docker build -f docker/Dockerfile.ml -t warrantygraph-ml:latest .
docker run --gpus all -v "$PWD":/workspace -w /workspace warrantygraph-ml:latest \
  python -m ml.fault_code_vision.train_ocr --manifest ... --require-cuda
```

### GAN lab (synthetic data factory)

Open [`fault_code_gan_synthetic_images.ipynb`](./fault_code_gan_synthetic_images.ipynb) — DCGAN/cGAN, phone augments, synthetic corpus.

### After real photos arrive

1. Add rows with `source=real`, freeze a test split.
2. Retrain reader.
3. Gate on `vision_full.real_photo_accuracy`.
4. Fill `models/finetunes/fault-code-ocr-vN.yaml`, pin `models/registry.yaml`.
5. Wire multipart API → `FaultCodeReader` → diagnose service.

---

## References

- Goodfellow et al. 2014 (GAN); Radford et al. 2016 (DCGAN); Mirza & Osindero 2014 (cGAN)
- Shorten & Khoshgoftaar 2019 (augmentation)
- This repo: `docs/sdd/09-PLATFORM-LLMOPS.md`, `docs/24-TBox-Sparse-Data-and-Enterprise-Scale-Patterns.md`
- Study module 20: synthetic data + MLOps + image ops (`study/seed_platform_modules.py`)

**Done when:** CUDA train job is reproducible, OCR is eval-gated, registry pin exists, and extracted codes boost GraphRAG the same way typed codes do today.